In [21]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os

import geopandas as gpd
import pandas as pd
from geoalchemy2 import Geometry

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [22]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def obtener_lotes_listos_para_procesar():
    engine = obtener_engine()
    
    # Consulta combinando ambas condiciones
    query = text("""
        SELECT * FROM siembra_surcado.data_lote 
        WHERE lineas_creadas = :estado_lineas 
          AND puntos_cargados = :estado_puntos
    """)
    
    try:
        with engine.connect() as conn:
            # Pasamos los parámetros booleanos de Python
            df = pd.read_sql(
                query, 
                conn, 
                params={
                    "estado_lineas": False, 
                    "estado_puntos": True
                }
            )
        return df
    except Exception as e:
        print(f"Error en la consulta: {e}")
        return None

In [23]:
# Uso
lotes_por_procesar = obtener_lotes_listos_para_procesar()

In [24]:
lotes_por_procesar

,id,unidad_01,unidad_02,unidad_05,campanha,fecha_de_registro,puntos_cargados,lineas_creadas,segmentos_creados,desviacion_calculada,velocidad_calculada
0,17,30,CAMPO DULCE,ER-L5-ER-L6,2026,2026-03-08,True,False,False,False,False
1,1,2238,EL PARAISO,L5,2026,2026-03-05,True,False,False,False,False
2,18,2238,EL PARAISO,L5-1,2026,2026-03-08,True,False,False,False,False
3,19,2238,EL PARAISO,L5-2,2026,2026-03-08,True,False,False,False,False
4,20,30,CAMPO DULCE,EP-L20-EP-L21,2026,2026-03-08,True,False,False,False,False
5,21,482,TEXAS,TX-1,2026,2026-03-08,True,False,False,False,False


In [25]:
def procesar_puntos_a_lineas():
    engine = obtener_engine()
    
    # 1. Obtenemos los IDs de los lotes pendientes (como en tu imagen)
    query_pendientes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE lineas_creadas = False AND puntos_cargados = True
    """)
    
    try:
        with engine.begin() as conn:  # Usamos .begin() para manejar la transacción
            lotes = conn.execute(query_pendientes).fetchall()
            
            if not lotes:
                print("No hay lotes pendientes por procesar.")
                return

            for lote in lotes:
                lote_id = lote[0]
                print(f"Procesando lote ID: {lote_id}...")

                # 2. Creamos la línea y la insertamos en lineas_completas
                # ST_MakeLine agrupa los puntos ordenados por isotime
                insert_query = text("""
                    INSERT INTO siembra_surcado.lineas_completas (geom, data_lote_id)
                    SELECT 
                        ST_MakeLine(geom ORDER BY isotime),
                        :lote_id
                    FROM siembra_surcado.detalles_lote_utm
                    WHERE lote_id = :lote_id;
                """)
                conn.execute(insert_query, {"lote_id": lote_id})

                # 3. Marcamos el lote como procesado
                update_query = text("""
                    UPDATE siembra_surcado.data_lote 
                    SET lineas_creadas = True 
                    WHERE id = :lote_id
                """)
                conn.execute(update_query, {"lote_id": lote_id})
                
                print(f"Lote {lote_id} finalizado con éxito.")

    except Exception as e:
        print(f"Ocurrió un error durante el procesamiento: {e}")

# Ejecutar el proceso
procesar_puntos_a_lineas()

Procesando lote ID: 17...
Lote 17 finalizado con éxito.
Procesando lote ID: 1...
Lote 1 finalizado con éxito.
Procesando lote ID: 18...
Lote 18 finalizado con éxito.
Procesando lote ID: 19...
Lote 19 finalizado con éxito.
Procesando lote ID: 20...
Lote 20 finalizado con éxito.
Procesando lote ID: 21...
Lote 21 finalizado con éxito.
